In [107]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [108]:
train_data_url = '/kaggle/input/competitions/titanic/train.csv'
test_data_url = '/kaggle/input/competitions/titanic/test.csv'
train_data = pd.read_csv(train_data_url)
test_data = pd.read_csv(test_data_url)

train_data.describe()
train_data.head()

# test_data.describe()
# test_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [109]:
y = train_data.Survived
train_data.drop(['Survived'], axis=1, inplace=True) # inplace-True modifies train_data directly


X_train_full, X_valid_full, y_train, y_valid = train_test_split(train_data, y, train_size=0.8, test_size=0.2, random_state=0)

cat_cols = [cols for cols in X_train_full.columns if X_train_full[cols].nunique() < 10 and X_train_full[cols].dtype == 'object']

num_cols = [cols for cols in X_train_full.columns if X_train_full[cols].dtype in ['int64', 'float64']]

tol_cols = num_cols + cat_cols
X_train = X_train_full[tol_cols].copy()
X_test = test_data[tol_cols].copy()
X_valid = X_valid_full[tol_cols].copy()

X_train.head()

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Sex,Embarked
140,141,3,NaN,0,2,15.2458,female,C
439,440,2,31.0,0,0,10.5000,male,S
817,818,2,31.0,1,1,37.0042,male,C
378,379,3,20.0,0,0,4.0125,male,C
491,492,3,21.0,0,0,7.2500,male,S


In [110]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

numerical_transformer = SimpleImputer(strategy='constant')

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one-hot-encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

model = XGBClassifier(random_state=0)

my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',model)
])
my_pipeline.fit(X_train,y_train)
my_predictions= my_pipeline.predict(X_valid)

mae = mean_absolute_error(my_predictions, y_valid)
print('Mean Absolute Error: '+str(mae))

print('Accuracy:', accuracy_score(y_valid, my_predictions))

predictions = my_pipeline.predict(X_test).round().astype(int)
output = pd.DataFrame({'PassengerId': X_test['PassengerId'],
                       'Survived': predictions})
output.to_csv('/kaggle/working/submission.csv', index=False)
print(output)


Mean Absolute Error: 0.1340782122905028
Accuracy: 0.8659217877094972
     PassengerId  Survived
0            892         0
1            893         0
2            894         0
3            895         0
4            896         0
..           ...       ...
413         1305         0
414         1306         1
415         1307         0
416         1308         0
417         1309         0

[418 rows x 2 columns]
